# Architecture C1 - Multi-Agent Always-L (Always Big)

This notebook runs Architecture C1, which is the "Always-Big" variant of Architecture C.

**Key difference from C (Adaptive)**:
- **C (Adaptive)**: Routes tasks to S/M/L developers based on story points
- **C1 (Always-L)**: Always uses Developer-L (strongest model) for all tasks

This enables comparison for **RQ3**: Can adaptive routing reduce cost while maintaining quality?

In [ ]:
import os
import sys
import subprocess
import pathlib

REPO_URL = "https://github.com/LLM4SE-group-15/ArchitecturesForCodeDevelopmentWithLLMs.git"
REPO_DIR = pathlib.Path("/content/ArchitecturesForCodeDevelopmentWithLLMs")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

print(f"Using repo at {REPO_DIR.resolve()}")

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import getpass
from huggingface_hub import login

#placeholder

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
try:
    user_info = api.whoami(token=os.environ["HF_TOKEN"])
    print("Logged in to Hugging Face as:", user_info.get("name") or user_info.get("user"))
except Exception as exc:
    print("Login check failed:", exc)

In [ ]:
import json
import time
import logging
import pathlib
import os

from datetime import datetime

DEFAULT_ROOT = pathlib.Path("/content/ArchitecturesForCodeDevelopmentWithLLMs")
ROOT = DEFAULT_ROOT if DEFAULT_ROOT.exists() else pathlib.Path.cwd()

sys.path.insert(0, str(ROOT))

LOG_DIR = ROOT / "log"
LOG_DIR.mkdir(exist_ok=True)

logger = logging.getLogger("architecture_C1")
logger.setLevel(logging.INFO)
if logger.handlers:
    logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_DIR / "architecture_C1.log")
stream_handler = logging.StreamHandler()
for handler in (file_handler, stream_handler):
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger ready. Repo root: %s", ROOT)
logger.info("Log files: %s", LOG_DIR)
print("Logs ->", LOG_DIR)

In [ ]:
import time
import json
import random
from src.data.task_loader import HumanEvalTaskLoader
from src.graph.graph import run_graph
from src.agents.llm import Architecture

ARCH = Architecture.C1
SEED_FIXED = 31

def run_humaneval_benchmark(limit: int = 15, shuffle: bool = True):
    """
    Run benchmark on HumanEval tasks.
    
    Args:
        limit: Number of tasks to run
        shuffle: If True, randomly sample tasks with fixed seed
    """
    loader = HumanEvalTaskLoader()
    all_tasks = loader.load_all()  # Load all 164 tasks
    
    if shuffle:
        random.seed(SEED_FIXED)
        tasks = random.sample(all_tasks, min(limit, len(all_tasks)))
    else:
        tasks = all_tasks[:limit]

    results = []
    total = len(tasks)
    logger.info("Loaded %s tasks from HumanEval (shuffle=%s, seed=%s)", total, shuffle, SEED_FIXED)
    print(f"Starting benchmark on {total} tasks...")
    
    for idx, task in enumerate(tasks, 1):
        logger.info("Running %s/%s %s", idx, total, task.task_id)
        print(f"[{idx}/{total}] Task {task.task_id} ({task.entry_point})... ", end="", flush=True)
        
        start = time.time()
        
        state = run_graph(
            task_id=task.task_id,
            task_description=task.prompt,
            test_code=task.test,
            entry_point=task.entry_point,
            architecture=ARCH,
        )

        elapsed = time.time() - start
        
        record = {
            "task_id": task.task_id,
            "entry_point": task.entry_point,
            "architecture": str(ARCH.value),
            "test_passed": state["test_passed"],
            "developer_tier": state.get("developer_tier"),
            "escalations": state["escalations"],
            "story_points_initial": state.get("story_points_initial"),
            "story_points_final": state.get("story_points_current"),
            "elapsed_seconds": elapsed,
        }
        results.append(record)
        
        logger.info(
            "Finished %s | pass=%s tier=%s escalations=%s elapsed=%.1fs",
            task.task_id,
            state["test_passed"],
            record["developer_tier"],
            record["escalations"],
            elapsed,
        )
        status_str = "PASS" if state["test_passed"] else "FAIL"
        print(f"{status_str} in {elapsed:.1f}s")

        with open(LOG_DIR / "architecture_C1.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")
            
    return results

# Run: 15 random tasks (seed=31 for reproducibility)
sample_results = run_humaneval_benchmark(limit=15, shuffle=True)

# Summary
passed_count = sum(1 for r in sample_results if r['test_passed'])
print(f"\nBenchmark Completed. Passed: {passed_count}/{len(sample_results)}")

In [ ]:
!cd log && cat architecture_C1.jsonl

## Evaluation Metrics for Architecture C1 (Always-L)

This section calculates the evaluation metrics as specified in `evaluation.md`:

- **Primary Metrics**: Pass Rate, Pass@1
- **Cost Metrics**: Execution Time, API Calls
- **C1 Specific**: Always uses tier L (no escalation expected)

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Load results from JSONL file
log_file = LOG_DIR / "architecture_C1.jsonl"
results = []
if log_file.exists():
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                results.append(json.loads(line))
else:
    print(f"Warning: {log_file} not found. Make sure to run the tasks first.")

df = pd.DataFrame(results)
print(f"Loaded {len(df)} task results")

# Show all rows
pd.set_option('display.max_rows', None)
df

In [ ]:
# Helper function for Wilson score confidence interval
def wilson_ci(successes, n, confidence=0.95):
    """Calculate Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0, 0, 0
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    p_hat = successes / n
    denominator = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denominator
    margin = z * np.sqrt((p_hat * (1 - p_hat) + z**2 / (4 * n)) / n) / denominator
    return p_hat, center - margin, center + margin

# Calculate Pass Rate (after all retries)
total_tasks = len(df)
passed_tasks = df["test_passed"].sum()
pass_rate = passed_tasks / total_tasks if total_tasks > 0 else 0

# For C1: Pass@1 = Pass Rate (always uses L, no escalation needed)
# But we still calculate it properly for consistency
passed_first_attempt = df[(df["test_passed"] == True) & (df["escalations"] == 0)].shape[0]
pass_at_1 = passed_first_attempt / total_tasks if total_tasks > 0 else 0

# Calculate 95% Confidence Intervals
_, ci_low, ci_high = wilson_ci(passed_tasks, total_tasks)
_, ci_low_at1, ci_high_at1 = wilson_ci(passed_first_attempt, total_tasks)

print("=" * 60)
print("PRIMARY METRICS - Architecture C1 (Always-L)")
print("=" * 60)
print(f"\nDataset: HumanEval (164 total problems)")
print(f"Tasks Evaluated: {total_tasks}")
print(f"\nPass Rate (after retries): {pass_rate:.2%} ({passed_tasks}/{total_tasks})")
print(f"  95% CI: [{ci_low:.2%}, {ci_high:.2%}]")
print(f"\nPass@1 (first attempt): {pass_at_1:.2%} ({passed_first_attempt}/{total_tasks})")
print(f"  95% CI: [{ci_low_at1:.2%}, {ci_high_at1:.2%}]")
print(f"\nNote: C1 always uses Developer-L, so escalations should be 0.")
print("=" * 60)

In [ ]:
# Calculate Cost Metrics
avg_time = df["elapsed_seconds"].mean()
std_time = df["elapsed_seconds"].std()
median_time = df["elapsed_seconds"].median()
total_time = df["elapsed_seconds"].sum()
min_time = df["elapsed_seconds"].min()
max_time = df["elapsed_seconds"].max()

# For C1: 3 API calls per task (Planner + Developer-L + Reviewer)
# No additional calls for escalation since we always use L
df["api_calls"] = 3 + (df["escalations"] * 2)
avg_api_calls = df["api_calls"].mean()
total_api_calls = df["api_calls"].sum()

print("\n" + "=" * 60)
print("COST METRICS")
print("=" * 60)
print(f"\nExecution Time Statistics:")
print(f"  Average Time per Task: {avg_time:.2f} seconds")
print(f"  Std Dev: {std_time:.2f} seconds")
print(f"  Median Time: {median_time:.2f} seconds")
print(f"  Min Time: {min_time:.2f} seconds")
print(f"  Max Time: {max_time:.2f} seconds")
print(f"  Total Execution Time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
print(f"\nAPI Calls:")
print(f"  Average API Calls per Task: {avg_api_calls:.2f}")
print(f"  Total API Calls: {total_api_calls}")
print("=" * 60)

In [ ]:
# Developer Tier Distribution (should all be L for C1)
print("\n" + "=" * 60)
print("DEVELOPER TIER DISTRIBUTION")
print("=" * 60)

if "developer_tier" in df.columns:
    tier_counts = df["developer_tier"].value_counts()
    print("\n| Developer Tier | Count | Percentage |")
    print("|----------------|-------|------------|")
    for tier in ["S", "M", "L"]:
        if tier in tier_counts.index:
            count = tier_counts[tier]
            pct = 100 * count / total_tasks
            print(f"| {tier}              | {count:>5} | {pct:>10.2f}% |")
        else:
            print(f"| {tier}              |     0 |       0.00% |")

# Escalation check (should be 0 for C1)
print(f"\nTotal Escalations: {df['escalations'].sum()} (expected: 0 for C1)")
print("=" * 60)

In [ ]:
# Per-Task Results
print("\n" + "=" * 60)
print("PER-TASK RESULTS")
print("=" * 60)
print("\n| Task ID           | Entry Point                  | Tier | Passed | Time (s) |")
print("|-------------------|------------------------------|------|--------|----------|")
for _, row in df.iterrows():
    status = "PASS" if row["test_passed"] else "FAIL"
    tier = row.get("developer_tier", "L")
    print(f"| {row['task_id']:<17} | {row['entry_point']:<28} | {tier:<4} | {status:<6} | {row['elapsed_seconds']:>8.2f} |")
print("=" * 60)

In [ ]:
# Visualizations
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 10)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Architecture C1 (Always-L) - HumanEval Evaluation Metrics", fontsize=16, fontweight="bold")

# 1. Pass/Fail Bar Chart
ax1 = axes[0, 0]
categories = ["Passed", "Failed"]
counts = [passed_tasks, total_tasks - passed_tasks]
colors = ["#2ecc71", "#e74c3c"]
bars = ax1.bar(categories, counts, color=colors, edgecolor="black")
ax1.set_ylabel("Number of Tasks")
ax1.set_title(f"Pass/Fail Distribution (Pass Rate: {pass_rate:.1%})")
for bar, count in zip(bars, counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f"{count} ({count/total_tasks:.1%})", ha="center", fontsize=11, fontweight="bold")

# 2. Pass Rate vs Pass@1 comparison
ax2 = axes[0, 1]
metrics = ["Pass Rate\n(after retries)", "Pass@1\n(first attempt)"]
values = [pass_rate * 100, pass_at_1 * 100]
colors_bar = ["#3498db", "#9b59b6"]
bars = ax2.bar(metrics, values, color=colors_bar, edgecolor="black")
ax2.set_ylabel("Percentage (%)")
ax2.set_title("Pass Rate vs Pass@1")
ax2.set_ylim(0, 100)
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f"{val:.1f}%", ha="center", fontsize=11, fontweight="bold")

# 3. Execution Time Box Plot (Passed vs Failed)
ax3 = axes[1, 0]
passed_times = df[df["test_passed"]]["elapsed_seconds"]
failed_times = df[~df["test_passed"]]["elapsed_seconds"]
if len(passed_times) > 0 and len(failed_times) > 0:
    box_data = [passed_times, failed_times]
    bp = ax3.boxplot(box_data, labels=["Passed", "Failed"], patch_artist=True)
    bp["boxes"][0].set_facecolor("#2ecc71")
    bp["boxes"][1].set_facecolor("#e74c3c")
else:
    ax3.bar(["Passed", "Failed"], [passed_times.mean() if len(passed_times) > 0 else 0, 
                                    failed_times.mean() if len(failed_times) > 0 else 0],
            color=["#2ecc71", "#e74c3c"])
ax3.set_ylabel("Execution Time (seconds)")
ax3.set_title("Execution Time by Result")

# 4. Execution Time Histogram
ax4 = axes[1, 1]
ax4.hist(df["elapsed_seconds"], bins=15, edgecolor="black", color="#3498db", alpha=0.7)
ax4.axvline(avg_time, color="red", linestyle="--", linewidth=2, label=f"Mean: {avg_time:.2f}s")
ax4.axvline(median_time, color="green", linestyle="--", linewidth=2, label=f"Median: {median_time:.2f}s")
ax4.set_xlabel("Execution Time (seconds)")
ax4.set_ylabel("Frequency")
ax4.set_title("Execution Time Distribution")
ax4.legend()

plt.tight_layout()
plt.savefig(LOG_DIR / "architecture_C1_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nVisualization saved to: {LOG_DIR / 'architecture_C1_metrics.png'}")

In [ ]:
# Summary Table
summary_data = {
    "Metric": [
        "Architecture",
        "Dataset",
        "Total Tasks",
        "Passed Tasks",
        "Pass Rate",
        "Pass@1",
        "95% CI Pass Rate",
        "95% CI Pass@1",
        "Avg Execution Time (s)",
        "Std Execution Time (s)",
        "Median Execution Time (s)",
        "Min Execution Time (s)",
        "Max Execution Time (s)",
        "Total Execution Time (s)",
        "Avg API Calls per Task",
        "Total API Calls",
        "Total Escalations",
        "Developer Tier Used",
    ],
    "Value": [
        "C1 (Always-L)",
        "HumanEval",
        total_tasks,
        int(passed_tasks),
        f"{pass_rate:.4f}",
        f"{pass_at_1:.4f}",
        f"[{ci_low:.4f}, {ci_high:.4f}]",
        f"[{ci_low_at1:.4f}, {ci_high_at1:.4f}]",
        f"{avg_time:.2f}",
        f"{std_time:.2f}",
        f"{median_time:.2f}",
        f"{min_time:.2f}",
        f"{max_time:.2f}",
        f"{total_time:.2f}",
        f"{avg_api_calls:.2f}",
        int(total_api_calls),
        int(df["escalations"].sum()),
        "Always L (Qwen2.5-Coder-32B)",
    ]
}

summary_df = pd.DataFrame(summary_data)

print("\n" + "=" * 60)
print("SUMMARY TABLE FOR ARCHITECTURE C1 (ALWAYS-L)")
print("=" * 60)
print(summary_df.to_string(index=False))
print("=" * 60)

# Save summary
summary_df.to_csv(LOG_DIR / "architecture_C1_summary.csv", index=False)

# Save detailed results
df.to_csv(LOG_DIR / "architecture_C1_detailed.csv", index=False)

print(f"\nSummary saved to: {LOG_DIR / 'architecture_C1_summary.csv'}")
print(f"Detailed results saved to: {LOG_DIR / 'architecture_C1_detailed.csv'}")